<a href="https://colab.research.google.com/github/awaiskhan005/DATA-SCIENCE-AND-AI-/blob/main/RESTAPI_USAGE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np


In [2]:
import joblib
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Load data
iris = load_iris()
X, y = iris.data, iris.target

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Train
clf = LogisticRegression(max_iter=200)
clf.fit(X_train, y_train)

# 4. Evaluate
preds = clf.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, preds))

# 5. Persist
joblib.dump(clf, "iris_model.joblib")
print("Model saved as iris_model.joblib")


Test accuracy: 1.0
Model saved as iris_model.joblib


In [4]:
!python train_model.py

python3: can't open file '/content/train_model.py': [Errno 2] No such file or directory


In [5]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import joblib
import numpy as np

# Load trained model
model = joblib.load("iris_model.joblib")

# FastAPI instance
app = FastAPI(title="Iris Classifier API")

# Request schema
class IrisSample(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.post("/predict")
async def predict(sample: IrisSample):
    try:
        data = np.array([[
            sample.sepal_length,
            sample.sepal_width,
            sample.petal_length,
            sample.petal_width
        ]])
        pred = model.predict(data)[0]
        label = iris.target_names[pred]
        return {"prediction": int(pred), "label": label}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


In [7]:
!uvicorn main:app --reload


INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [1528] using StatReload
ERROR:    Error loading ASGI app. Could not import module "main".
INFO:     Stopping reloader process [1528]


In [8]:
!pip install scikit-learn fastapi uvicorn pyngrok nest_asyncio


In [9]:
import joblib
import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

from pyngrok import ngrok
import nest_asyncio


In [10]:
# 1. Load data
iris = load_iris()
X, y = iris.data, iris.target

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Train
clf = LogisticRegression(max_iter=200)
clf.fit(X_train, y_train)

# 4. Evaluate
preds = clf.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, preds):.2%}")

# 5. Save
joblib.dump(clf, "iris_model.joblib")
print("Model saved as iris_model.joblib")


Test accuracy: 100.00%
Model saved as iris_model.joblib


In [11]:
# Allow Uvicorn to run inside Colab
nest_asyncio.apply()

# Load the saved model
model = joblib.load("iris_model.joblib")

# FastAPI setup
app = FastAPI(title="Iris Classifier API")

# Pydantic schema for incoming JSON
class IrisSample(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.post("/predict")
async def predict(sample: IrisSample):
    try:
        arr = np.array([[
            sample.sepal_length,
            sample.sepal_width,
            sample.petal_length,
            sample.petal_width
        ]])
        pred = model.predict(arr)[0]
        label = iris.target_names[pred]
        return {"prediction": int(pred), "label": label}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


In [13]:
from pyngrok import ngrok

# replace with your real token
ngrok.set_auth_token("2zih76IzJfdGK97J8cjVTdizJzB_4VBroHhNzU2BFgkQu2TsB")


In [14]:
public_url = ngrok.connect(8000).public_url
print("🔥 API live at:", public_url)


🔥 API live at: https://1ba4ecd2e592.ngrok-free.app


In [15]:
from fastapi.testclient import TestClient

client = TestClient(app)

# Example request
resp = client.post("/predict", json={
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2
})
print(resp.status_code, resp.json())


200 {'prediction': 0, 'label': 'setosa'}


In [16]:
@app.get("/")
async def read_root():
    return {"message": "Iris Classifier API is up and running!"}
